# Content Ranking Eval Pipeline

**Can a text-quality rubric predict engagement on a ranked content surface?**

Rachel McCaig (Barden) · June 2026 · [GitHub](https://github.com/rachelmccaig/content-ranking-eval)

---

This notebook is a walkthrough of the evaluation design behind the repo. It mirrors the README: define a content-quality rubric, score Hacker News stories with a Claude model-as-judge, compare the judge ranking against observed engagement, and inspect where the two diverge.

**The core question:** does a text-based quality rubric measure the same construct as community engagement?

**The finding:** across 8 total runs, including two statistically significant 499-story runs, the answer was no. The relationship was small, negative, and stable. The interesting result is not that the metric went down; it is that the eval exposed a construct validity gap.

Cells with baked outputs show representative results from actual API runs. The heuristic demo cell is runnable without an API key.


## The Rubric

Four dimensions scored 1-10 by a Claude model-as-judge:

| Dimension | Weight | What it measures |
|---|---:|---|
| `title_clarity` | 20% | Is the title clear and understandable? |
| `specificity` | 25% | Specific and concrete vs. vague or generic? |
| `topic_novelty` | 30% | Fresh or non-obvious? Penalizes rehashed takes. |
| `information_density` | 25% | Real signal vs. clickbait? |

**Removed dimension:** `engagement_potential` appeared in the first version. It was circular: using "how likely is this to get engagement" to predict engagement encodes the answer in the rubric. It was removed and replaced with `topic_novelty`.

**Rejected dimension:** `community_identity` was considered and deliberately left out. Community identity is not a property of the title text. It requires behavioral data: author history, domain reputation, topic momentum, and audience context. Asking the model to infer that from text alone would produce plausible but ungrounded scores.

**Ground truth:** composite engagement = `hn_score + (comments x 2)`. Comments require more active intent than upvotes, so they are weighted more heavily.


In [1]:
WEIGHTS = {
    "title_clarity":       0.20,
    "specificity":         0.25,
    "topic_novelty":       0.30,
    "information_density": 0.25,
}
COMMENT_WEIGHT = 2

def composite(scores: dict) -> float:
    return sum(scores[dim] * weight for dim, weight in WEIGHTS.items())

print("Rubric weights:")
for dim, weight in WEIGHTS.items():
    print(f"  {dim:<22} {weight:.0%}")
print(f"\nGround truth: hn_score + (comments × {COMMENT_WEIGHT})")

Rubric weights:
  title_clarity          20%
  specificity            25%
  topic_novelty          30%
  information_density    25%

Ground truth: hn_score + (comments × 2)


## Demo: heuristic scorer (no API key needed)

To show the pipeline shape without an API call, a rule-based heuristic scorer uses title length, colons, numbers, and question marks as proxies for the rubric dimensions.

The examples below were chosen to illustrate the construct gap: titles the judge reliably rates high vs. titles HN reliably rewards.

In [2]:
def score_heuristic(title: str) -> dict:
    words = title.split()
    wc = len(words)
    has_number = any(w.replace(",", "").replace(".", "").isdigit() for w in words)
    has_colon  = ":" in title
    is_q       = title.endswith("?")
    return {
        "title_clarity":       min(10, max(1, 5 + (2 if has_colon else 0) + (1 if wc < 12 else -1))),
        "specificity":         min(10, max(1, 4 + (3 if has_number else 0) + (2 if has_colon else 0))),
        "topic_novelty":       min(10, max(1, 5 + (2 if not is_q else 0) + (1 if has_number else 0))),
        "information_density": min(10, max(1, min(wc, 10) - (2 if is_q else 0))),
    }

examples = [
    "Building a Custom Memory Allocator in C++: A Complete Guide",
    "A Deep Dive into PostgreSQL's VACUUM Process",
    "Running local models is good now",
    "Ask HN: What are you working on?",
    "Nipkow Disk Mechanical TV Simulator",
    "Iroh 1.0",
]

print("Heuristic scores for representative titles:\n")
scored = []
for title in examples:
    s = score_heuristic(title)
    c = composite(s)
    scored.append((c, title))
    print(f"Title: {title}")
    print(f"  clarity={s['title_clarity']}  specificity={s['specificity']}  "
          f"novelty={s['topic_novelty']}  density={s['information_density']}   "
          f"→ composite {c:.2f}\n")

scored.sort(reverse=True)
print("Judge ranking (heuristic):")
for i, (c, t) in enumerate(scored, 1):
    print(f"  #{i}  [{c:.2f}]  {t}")

print("\nActual HN engagement ranking (from live runs):")
hn_order = [
    ("Iroh 1.0",                                          "← judge #4"),
    ("Ask HN: What are you working on?",                  "← judge #3"),
    ("Running local models is good now",                  "← judge #5"),
    ("Nipkow Disk Mechanical TV Simulator",               "← judge #6"),
    ("A Deep Dive into PostgreSQL's VACUUM Process",      "← judge #2"),
    ("Building a Custom Memory Allocator in C++...",      "← judge #1"),
]
for i, (t, note) in enumerate(hn_order, 1):
    print(f"  #{i}  {t:<52} {note}")
print("\nThe judge and HN are nearly reversed on this set.")

Heuristic scores for representative titles:

Title: Building a Custom Memory Allocator in C++: A Complete Guide
  clarity=8  specificity=6  novelty=7  density=10  → composite 7.70

Title: A Deep Dive into PostgreSQL's VACUUM Process
  clarity=6  specificity=4  novelty=7  density=7   → composite 6.05

Title: Running local models is good now
  clarity=6  specificity=4  novelty=7  density=6   → composite 5.80

Title: Ask HN: What are you working on?
  clarity=8  specificity=6  novelty=5  density=5   → composite 5.85

Title: Nipkow Disk Mechanical TV Simulator
  clarity=6  specificity=4  novelty=7  density=5   → composite 5.55

Title: Iroh 1.0
  clarity=6  specificity=7  novelty=8  density=2   → composite 5.85

Judge ranking (heuristic):
  #1  [7.70]  Building a Custom Memory Allocator in C++: A Complete Guide
  #2  [6.05]  A Deep Dive into PostgreSQL's VACUUM Process
  #3  [5.85]  Ask HN: What are you working on?
  #4  [5.85]  Iroh 1.0
  #5  [5.80]  Running local models is good now
  #6  

## Evolution of the Evaluation

The experiment history is collapsed to methodology-changing steps, matching the README.

| Version | Change | Why it mattered |
|---|---|---|
| v1 | Initial rubric included `engagement_potential` | Circular predictor: it encoded the target outcome |
| v2 | Replaced `engagement_potential` with `topic_novelty` | Kept the rubric focused on content-quality inputs |
| v3 | Changed ground truth from raw upvotes to `upvotes + comments x 2` | Comments are a stronger chosen-interaction signal |
| v3.1 | Added story age and title length as context | Tested whether simple metadata explained the gap |
| v3.2 | Ran larger 499-story samples with Haiku | Confirmed the negative correlation was statistically significant |

The correlation stayed negative across these changes. That pushed the diagnosis away from "bad prompt" and toward construct validity: the rubric was measuring text quality, while HN engagement rewarded community resonance.


In [3]:
results = [
    ("Heuristic baseline", "none",  50,  -0.245, None,  "Rule-based scorer, no API"),
    ("Initial LLM judge", "Opus",  50,  -0.021, None,  "Included engagement_potential, later identified as circular"),
    ("Revised rubric", "Opus",  50,  -0.099, None,  "Replaced circular predictor with topic_novelty"),
    ("Composite engagement", "Opus",  50,  -0.234, None,  "Used upvotes + weighted comments as target"),
    ("Title length added", "Opus",  50,  -0.159, 0.273, "Added title_word_count"),
    ("Larger run", "Haiku", 499, -0.113, 0.012, "First statistically significant result"),
    ("Repeat larger run", "Haiku", 499, -0.104, 0.022, "Finding held across another sample"),
]

print("Methodology-changing results:\n")
print(f"{'Run':<22} {'Model':<8} {'N':<6} {'Corr':<9} {'p':<8} {'Notes'}")
print("-" * 104)
for run, model, n, corr, p, notes in results:
    sig = "*" if p is not None and p < 0.05 else " "
    p_str = f"{p:.3f}" if p is not None else "-"
    print(f"{run:<22} {model:<8} {n:<6} {corr:<9.3f} {p_str:<8} {sig}{notes}")

print("\n* p < 0.05")
print("\nAcross 8 total runs, the conclusion stayed stable.")
print("The construct gap is structural: more capable models did not close it; better signals would.")


Methodology-changing results:

Run                    Model    N      Corr      p        Notes
--------------------------------------------------------------------------------------------------------
Heuristic baseline     none     50     -0.245    -         Rule-based scorer, no API
Initial LLM judge      Opus     50     -0.021    -         Included engagement_potential, later identified as circular
Revised rubric         Opus     50     -0.099    -         Replaced circular predictor with topic_novelty
Composite engagement   Opus     50     -0.234    -         Used upvotes + weighted comments as target
Title length added     Opus     50     -0.159    0.273     Added title_word_count
Larger run             Haiku    499    -0.113    0.012    *First statistically significant result
Repeat larger run      Haiku    499    -0.104    0.022    *Finding held across another sample

* p < 0.05

Across 8 total runs, the conclusion stayed stable.
The construct gap is structural: more capable mode

In [4]:
disagreements = [
    {
        "title":   "Claude Fable 5",
        "judge":   473, "eng": 7, "gap": 466,
        "why":     ["Product launch for a model with massive community following.",
                    "Judge sees two words. HN sees a community event."],
    },
    {
        "title":   "Iroh 1.0",
        "judge":   465, "eng": 16, "gap": 449,
        "why":     ["Version bump for a project with a devoted community.",
                    "Same pattern — judge sees an uninformative title."],
    },
    {
        "title":   "Ask HN: What are you working on? (June 2026)",
        "judge":   454, "eng": 6, "gap": 448,
        "why":     ["Monthly ritual. Judge correctly scores it low-info.",
                    "HN shows up anyway. Community behavior, not content quality."],
    },
    {
        "title":   "I admire Fabrice Bellard",
        "judge":   450, "eng": 3, "gap": 447,
        "why":     ["Hero worship. Judge correctly penalizes it for low information",
                    "density. HN does not care."],
    },
    {
        "title":   "How to earn a billion dollars",
        "judge":   361, "eng": 5, "gap": 356,
        "why":     ["paulgraham.com. Judge correctly flags it as vague and",
                    "potentially clickbait. Judge can't see the domain authority."],
    },
]

print("Biggest disagreements — n=499 run (Haiku, Spearman -0.104, p=0.022)")
print("═" * 79)
for d in disagreements:
    print(f"\nJudge underrated by {d['gap']} positions")
    print(f"  Title:       {d['title']}")
    print(f"  Judge #{d['judge']}   Engagement #{d['eng']}")
    print(f"  Why:         {d['why'][0]}")
    print(f"               {d['why'][1]}")

print("\n" + "═" * 79)
print("These same titles appear as top disagreements across nearly every run.")
print("At n=499 they are not edge cases — they are a taxonomy.")

Biggest disagreements — n=499 run (Haiku, Spearman -0.104, p=0.022)
═══════════════════════════════════════════════════════════════════════════════

Judge underrated by 466 positions
  Title:       Claude Fable 5
  Judge #473   Engagement #7
  Why:         Product launch for a model with massive community following.
               Judge sees two words. HN sees a community event.

Judge underrated by 449 positions
  Title:       Iroh 1.0
  Judge #465   Engagement #16
  Why:         Version bump for a project with a devoted community.
               Same pattern — judge sees an uninformative title.

Judge underrated by 448 positions
  Title:       Ask HN: What are you working on? (June 2026)
  Judge #454   Engagement #6
  Why:         Monthly ritual. Judge correctly scores it low-info.
               HN shows up anyway. Community behavior, not content quality.

Judge underrated by 447 positions
  Title:       I admire Fabrice Bellard
  Judge #450   Engagement #3
  Why:         Hero worsh

## What This Means

The negative correlation is a **construct validity finding**, not just a disappointing metric.

The judge was consistent. It rewarded clear, specific, information-dense titles. HN engagement rewarded something else: community resonance, breaking news cycles, domain authority, product launches with known followings, and recurring rituals.

Three failure modes showed up repeatedly:

**1. Community resonance** — Short titles for projects with devoted followings, such as `Iroh 1.0` or `Claude Fable 5`. The judge sees an uninformative title. HN sees a community event.

**2. Breaking news cycles** — The judge has no topic-momentum signal. Sparse titles can perform well when they are part of an active news cycle.

**3. Domain authority** — `paulgraham.com` attached to a vague title is a quality guarantee to HN. The judge sees the words; HN sees the source.

The next lever is not more prompt tuning. It is better behavioral signal: author/domain history, topic trend data, audience context, and a human-scored golden set for known failure modes.


## Engineering Review and Future Direction

After the core iteration, the script was reviewed with Codex for engineering hardening. That review identified useful robustness improvements: tie-aware rank handling, LLM output validation, retry/backoff, checkpointing for long runs, optional age-adjusted engagement, and source selection across `topstories`, `newstories`, and `beststories`.

Those changes were kept separate from the canonical script because some alter the methodology used for the reported results. The main repo preserves the script that generated the README findings.

The strongest future experiment is a T0 prediction design:

1. Sample stories from `newstories`.
2. Score them before HN's ranking algorithm has amplified visibility.
3. Wait 6-24 hours.
4. Collect engagement.
5. Test whether the judge predicts future engagement.

That would reduce the feedback loop where HN's own ranking system affects visibility, which affects engagement, which then becomes the ground truth.
